# Classical AI Logical Reasoning System

**Rahul Kumar Bhikundia | Roll No.: M26CSA006 | IIT Jodhpur**

This notebook checks whether a query follows from a knowledge base.

| Dataset | Input | Method |
|---|---|---|
| RuleTaker | Context, statement | Forward chaining |
| CounterBench | Given information, question | Causal interventions |
| FOLIO | FOL premises, conclusion | Grounding and DPLL |
| General NL | Natural-language sentences | Rule-based reasoning |

**Pipeline:** Input → Parsing → Logic representation → Reasoning → Prediction

Run the cells in order. Place `.jsonl` files in `Data/` or beside the notebook. The final cell evaluates `dataset_files` and saves `submission.csv`.

**Assumptions:** Undeduced queries are `False`; FOLIO `Uncertain` labels count as not entailed. NL parsing uses fixed templates, not a language model.


## 1. Internal Representation

- `Term`: constant or variable.
- `Atom`: predicate, arguments and polarity.
- `HornRule`: premises imply a conclusion.


In [13]:
import json
import time
import re
import os
import heapq
from difflib import SequenceMatcher
from itertools import product
import pandas as pd

# =====================================================================
# 1. INTERNAL REPRESENTATION DATA STRUCTURES
# =====================================================================

class Term:
    """Represents a logical term: either a Variable (?x) or a Constant (bob)."""
    def __init__(self, name, is_var=False):
        self.name = str(name).strip()
        self.is_var = is_var

    def __repr__(self):
        return f"?{self.name}" if self.is_var else self.name

    def __eq__(self, other):
        return isinstance(other, Term) and self.name == other.name and self.is_var == other.is_var

    def __hash__(self):
        return hash((self.name, self.is_var))


class Atom:
    """Represents a First-Order or Propositional Atom: e.g., Smart(bob) or ~Plant(x)."""
    def __init__(self, predicate, args=None, negated=False):
        self.predicate = str(predicate).strip().lower()
        self.args = tuple(args) if args else ()
        self.negated = bool(negated)

    def negate(self):
        return Atom(self.predicate, self.args, not self.negated)

    def is_ground(self):
        return all(not arg.is_var for arg in self.args)

    def substitute(self, theta):
        new_args = [theta.get(arg, arg) for arg in self.args]
        return Atom(self.predicate, new_args, self.negated)

    def __repr__(self):
        prefix = "¬" if self.negated else ""
        if not self.args:
            return f"{prefix}{self.predicate}"
        args_str = ", ".join( repr(a) for a in self.args )
        return f"{prefix}{self.predicate}({args_str})"

    def __eq__(self, other):
        return (isinstance(other, Atom) and
                self.predicate == other.predicate and
                self.args == other.args and
                self.negated == other.negated)

    def __hash__(self):
        return hash((self.predicate, self.args, self.negated))


class HornRule:
    """Represents a definite/general rule: premise_1 ^ ... ^ premise_k -> conclusion."""
    def __init__(self, premises, conclusion, raw_text=""):
        self.premises = list(premises)
        self.conclusion = conclusion
        self.raw_text = raw_text

    def __repr__(self):
        hyps = " ∧ ".join(repr(p) for p in self.premises)
        return f"({hyps} → {repr(self.conclusion)})"

## 2. Unification and Rule Reasoning

`unify_atoms` matches atoms and returns a consistent variable substitution.

`RuleReasoningEngine` provides:
- `forward_chaining(goal)`: applies rules until the goal is found or no facts change.
- `backward_chaining(goal)`: works backward from the goal and prevents cycles.
- `a_star_proof_search(goal)`: searches possible proofs using unresolved subgoals as its heuristic.

Proof steps are stored in `engine.trace`.


In [14]:
# =====================================================================
# 2. UNIFICATION, FORWARD CHAINING, BACKWARD CHAINING & A* SEARCH
# =====================================================================

def unify_atoms(atom1, atom2, theta=None):
    """Classical Robinson Unification for two First-Order Logic atoms."""
    if theta is None:
        theta = {}
    if atom1.predicate != atom2.predicate or atom1.negated != atom2.negated:
        return None
    if len(atom1.args) != len(atom2.args):
        return None

    theta = dict(theta)
    for t1, t2 in zip(atom1.args, atom2.args):
        t1 = theta.get(t1, t1)
        t2 = theta.get(t2, t2)
        if t1 == t2:
            continue
        elif t1.is_var:
            theta[t1] = t2
        elif t2.is_var:
            theta[t2] = t1
        else:
            return None
    return theta


class RuleReasoningEngine:
    """
    Implements Classical AI Reasoning for Rule-Based KBs:
    1. Forward Chaining (Data-driven fixed-point inference)
    2. Backward Chaining (Goal-driven DFS with unification & cycle detection)
    3. A* Search (Heuristic shortest proof-tree search)
    """
    def __init__(self, facts=None, rules=None):
        self.facts = set(facts) if facts else set()
        self.rules = list(rules) if rules else []
        self.trace = []

    def forward_chaining(self, goal_atom):
        """Classical Forward Chaining with Unification."""
        known_facts = set(self.facts)
        self.trace = [f"Initial Facts: {known_facts}"]
        if goal_atom in known_facts:
            self.trace.append(f"Goal {goal_atom} already in initial facts.")
            return True

        added = True
        while added:
            added = False
            for rule in self.rules:
                # Match rule premises against current known ground facts
                substitutions = self._match_premises(rule.premises, known_facts, {})
                for theta in substitutions:
                    inferred = rule.conclusion.substitute(theta)
                    if inferred not in known_facts:
                        known_facts.add(inferred)
                        added = True
                        prem_str = " ∧ ".join(repr(p.substitute(theta)) for p in rule.premises)
                        self.trace.append(f"Derived {inferred} from {prem_str} via {rule}")
                        if inferred == goal_atom:
                            return True
        return goal_atom in known_facts

    def _match_premises(self, premises, known_facts, theta):
        if not premises:
            return [theta]
        first = premises[0].substitute(theta)
        rest = premises[1:]
        results = []
        for fact in known_facts:
            new_theta = unify_atoms(first, fact, theta)
            if new_theta is not None:
                results.extend(self._match_premises(rest, known_facts, new_theta))
        return results

    def backward_chaining(self, goal_atom, visited=None):
        """Classical Goal-Driven Backward Chaining (DFS) with cycle detection."""
        if visited is None:
            visited = set()
        if goal_atom in self.facts:
            return True
        if goal_atom in visited:
            return False
        visited.add(goal_atom)

        for rule in self.rules:
            theta = unify_atoms(rule.conclusion, goal_atom, {})
            if theta is not None:
                subgoals = [p.substitute(theta) for p in rule.premises]
                if self._prove_conjunction_bc(subgoals, visited.copy()):
                    return True
        return False

    def _prove_conjunction_bc(self, subgoals, visited):
        if not subgoals:
            return True
        first = subgoals[0]
        rest = subgoals[1:]
        if first.is_ground():
            if self.backward_chaining(first, visited.copy()):
                return self._prove_conjunction_bc(rest, visited)
            return False
        else:
            # Instantiate unbound variables using known constants
            constants = {arg for f in self.facts for arg in f.args if not arg.is_var}
            for const in constants:
                var = next(a for a in first.args if a.is_var)
                theta = {var: const}
                if self.backward_chaining(first.substitute(theta), visited.copy()):
                    if self._prove_conjunction_bc([g.substitute(theta) for g in rest], visited):
                        return True
            return False

    def a_star_proof_search(self, goal_atom):
        """
        A* Heuristic Search to find the minimum-step derivation of goal_atom.
        State: frozenset of subgoals remaining to be proved.
        g(n): number of rule applications so far.
        h(n): number of remaining unproven subgoals not in self.facts.
        """
        initial_unproven = frozenset([] if goal_atom in self.facts else [goal_atom])
        if not initial_unproven:
            return True, 0

        # Priority Queue: (f = g + h, g, state_frozenset)
        pq = [(len(initial_unproven), 0, initial_unproven)]
        visited = {initial_unproven: 0}

        while pq:
            f, g, current_goals = heapq.heappop(pq)
            if not current_goals:
                return True, g

            # Select first unproven goal to expand
            target = next(iter(current_goals))
            remaining = set(current_goals) - {target}

            for rule in self.rules:
                theta = unify_atoms(rule.conclusion, target, {})
                if theta is not None:
                    new_subgoals = {
                        p.substitute(theta)
                        for p in rule.premises
                        if p.substitute(theta) not in self.facts
                    }
                    # Ground any unbound variables if needed
                    if all(sg.is_ground() for sg in new_subgoals):
                        next_state = frozenset(remaining | new_subgoals)
                        new_g = g + 1
                        if next_state not in visited or new_g < visited[next_state]:
                            visited[next_state] = new_g
                            h = len(next_state)
                            heapq.heappush(pq, (new_g + h, new_g, next_state))
        return False, -1

## 3. CounterBench: Counterfactual Reasoning

The engine changes a variable through an intervention (`do(X = x)`) and predicts the queried variable.

- `A causes B` → `B := A`
- `A and B cause C` → `C := A ∧ B`
- `A or not B causes C` → `C := A ∨ ¬B`

Observed parentless variables keep their values; other parentless variables default to `False`. Equations are evaluated until values stop changing.


In [15]:
# =====================================================================
# 3. STRUCTURAL CAUSAL MODEL & COUNTERFACTUAL ENGINE (COUNTERBENCH)
# =====================================================================

class CausalGraphEngine:
    """
    Classical Rung-3 Counterfactual Reasoning Engine using Pearl's 3-Step Procedure:
    1. Abduction: Fix observed exogenous variables ('We observed ...') to their observed values.
    2. Action: Apply do(X = x) counterfactual interventions to structural equations.
    3. Prediction: Forward-propagate structural assignments to evaluate target query.
    """
    def __init__(self):
        # Maps target_var -> list of (op_type, [(var, is_negated), ...], target_negated)
        self.equations = {}
        self.all_vars = set()
        self.observations = {}
        self.root_actual = None

    @staticmethod
    def _clean_var(token):
        token = token.strip().strip(".,;").lower()
        token = re.sub(r"^(?:we know that|and|together)\s+", "", token).strip()
        if token.startswith("not "):
            return token[4:].strip(), True
        return token, False

    def parse_counterbench(self, given_info, question):
        # 1. Extract observations if present
        obs_match = re.search(r"We observed\s+([^\.]+)", given_info, re.IGNORECASE)
        causal_text = given_info
        if obs_match:
            obs_str = obs_match.group(1).strip()
            causal_text = given_info[:obs_match.start()].strip()
            for item in re.split(r"\s+and\s+|,\s*", obs_str):
                if item.strip():
                    v, neg = self._clean_var(item)
                    if v:
                        self.observations[v] = not neg
                        self.all_vars.add(v)

        # 2. Extract causal equations from causal_text
        causal_text = re.sub(r"^We know that\s+", "", causal_text, flags=re.IGNORECASE).strip().rstrip(".")
        raw_segments = [s.strip() for s in causal_text.split(",") if s.strip()]

        # Reconstruct clauses where a single 'causes' has comma-separated targets
        clauses = []
        last_lhs = None
        for seg in raw_segments:
            if re.search(r"\bcauses?\b", seg, re.IGNORECASE):
                clauses.append(seg)
                m = re.split(r"\bcauses?\b", seg, maxsplit=1, flags=re.IGNORECASE)
                last_lhs = m[0].strip()
            elif last_lhs is not None:
                # Continuation of RHS targets from previous LHS (e.g., "Sline, Frim, and Qado")
                clean_rhs = re.sub(r"^and\s+", "", seg, flags=re.IGNORECASE).strip()
                clauses.append(f"{last_lhs} causes {clean_rhs}")

        for clause in clauses:
            clause = re.sub(r"^and\s+", "", clause, flags=re.IGNORECASE).strip()
            parts = re.split(r"\s+together\s+causes?\s+|\s+causes?\s+", clause, maxsplit=1, flags=re.IGNORECASE)
            if len(parts) != 2:
                continue
            lhs_str, rhs_str = parts[0].strip(), parts[1].strip()
            lhs_str = re.sub(r"\s+together$", "", lhs_str, flags=re.IGNORECASE).strip()

            # Determine LHS operator (AND vs OR)
            if re.search(r"\s+or\s+", lhs_str, re.IGNORECASE):
                op = "OR"
                lhs_tokens = re.split(r"\s+or\s+", lhs_str, flags=re.IGNORECASE)
            else:
                op = "AND"
                lhs_tokens = re.split(r"\s+and\s+|,\s*", lhs_str, flags=re.IGNORECASE)

            lhs_literals = []
            for tok in lhs_tokens:
                if tok.strip():
                    v, neg = self._clean_var(tok)
                    lhs_literals.append((v, neg))
                    self.all_vars.add(v)

            # Parse RHS targets (may contain "A and B")
            rhs_tokens = re.split(r"\s+and\s+", rhs_str, flags=re.IGNORECASE)
            for rtok in rhs_tokens:
                if rtok.strip():
                    tv, tneg = self._clean_var(rtok)
                    self.all_vars.add(tv)
                    self.equations.setdefault(tv, []).append((op, lhs_literals, tneg))

        # 3. Parse Question for Interventions and Target
        interventions = {}
        target_var = None

        # Pattern A: "Would X occur if Y instead of Z?" or "Would X occur if Y and W?"
        m1 = re.search(r"Would\s+([A-Za-z0-9_]+)\s+occur\s+if\s+(.+?)(?:\s+instead of\s+([A-Za-z0-9_]+))?\?", question, re.IGNORECASE)
        if m1:
            target_var = m1.group(1).strip().lower()
            int_str = m1.group(2).strip()
            if m1.group(3):
                self.root_actual = m1.group(3).strip().lower()
            for part in re.split(r"\s+and\s+", int_str, flags=re.IGNORECASE):
                v, neg = self._clean_var(part)
                interventions[v] = not neg
                self.all_vars.add(v)
                if self.root_actual is None:
                    self.root_actual = v

        # Pattern B: "Assume X, and based on this assumption, further suppose Y. Would Z occur?"
        m2 = re.search(r"Assume\s+(.+?),\s*and based on this assumption,\s*further suppose\s+(.+?)\.\s*Would\s+([A-Za-z0-9_]+)\s+occur\?", question, re.IGNORECASE)
        if m2:
            v1, neg1 = self._clean_var(m2.group(1))
            v2, neg2 = self._clean_var(m2.group(2))
            target_var = m2.group(3).strip().lower()
            interventions[v1] = not neg1
            interventions[v2] = not neg2
            self.root_actual = v1
            self.all_vars.update([v1, v2, target_var])

        return interventions, target_var

    def _evaluate_world(self, fixed_assignments):
        """Forward-propagates structural equations to a fixed point."""
        state = {v: False for v in self.all_vars}
        state.update(fixed_assignments)

        for _ in range(len(self.all_vars) + 2):
            changed = False
            for tv, rules in self.equations.items():
                if tv in fixed_assignments:
                    continue
                # Evaluate rules for target variable tv
                val = False
                for op, lits, tneg in rules:
                    if op == "AND":
                        lhs_val = all((not state[v] if neg else state[v]) for v, neg in lits)
                    else:
                        lhs_val = any((not state[v] if neg else state[v]) for v, neg in lits)
                    rule_val = (not lhs_val) if tneg else lhs_val
                    val = val or rule_val
                if state[tv] != val:
                    state[tv] = val
                    changed = True
            if not changed:
                break
        return state

    def solve(self, given_info, question):
        interventions, target_var = self.parse_counterbench(given_info, question)
        if not target_var:
            return False

        # Step 1: Abduction - Identify exogenous variables (no incoming causal equations)
        exogenous = [v for v in self.all_vars if v not in self.equations]
        base_fixed = {}

        # Any directly observed exogenous variable is fixed to its observed value
        for v, obs_val in self.observations.items():
            if v in exogenous:
                base_fixed[v] = obs_val

        # Step 2: Action - Apply do(interventions)
        cf_fixed = dict(base_fixed)
        cf_fixed.update(interventions)

        # Step 3: Prediction - Evaluate counterfactual world
        cf_state = self._evaluate_world(cf_fixed)
        return bool(cf_state.get(target_var, False))

## 4. FOLIO: Bounded DPLL Prover

Entailment is tested by checking whether `KB ∧ ¬Conclusion` is unsatisfiable.

The prover parses FOL, aligns unmatched vocabulary, grounds quantifiers (up to 3 constants; 2 when nested), converts to CNF using Tseitin transformation, then runs DPLL with unit propagation.

DPLL is limited to 100 nodes. Reaching the limit is treated as satisfiable, so some true entailments may be missed.


In [16]:
# =====================================================================
# 4. FAST FIRST-ORDER LOGIC PARSER & BOUNDED DPLL PROVER (FOLIO)
# =====================================================================

class FOLNode:
    def __init__(self, kind, val=None, children=None, var=None):
        self.kind = kind          # 'ATOM', 'NOT', 'AND', 'OR', 'IMP', 'IFF', 'XOR', 'ALL', 'EXISTS'
        self.val = val            # (pred_name, (arg1, arg2, ...)) for ATOM
        self.children = children or []
        self.var = var


class FOLIOProver:
    """
    Fast Classical FOL Theorem Prover for FOLIO:
    1. Parses Unicode FOL formulas into an AST.
    2. Aligns conclusion predicates/constants with KB vocabulary.
    3. Dynamically bounds Herbrand domain size by quantifier depth.
    4. Uses Memoized Tseitin CNF + Step-Bounded DPLL branching only on ground atoms.
    """
    @staticmethod
    def clean_fol_line(line):
        line = re.sub(r"(?:Never:|Simeng:|\[).*$", "", line).strip()
        line = line.replace("—>", "→").replace("->", "→").replace("↔", "↔").replace("⟷", "↔")
        line = re.sub(r"\b([A-Za-z0-9_.-]+)\s*=\s*([A-Za-z0-9_.-]+)\b", r"Equal(\1, \2)", line)
        open_c, close_c = line.count("("), line.count(")")
        if open_c > close_c:
            line += ")" * (open_c - close_c)
        elif close_c > open_c:
            while line.endswith(")") and line.count(")") > line.count("("):
                line = line[:-1]
        return line

    @classmethod
    def tokenize(cls, text):
        text = cls.clean_fol_line(text)
        token_spec = re.compile(
            r"\s*(?:(∀|∃)\s*([a-zA-Z0-9_]+)|(→|↔|⊕|∧|∨|¬|\(|\)|,)|([A-Za-z0-9_.\u00C0-\u024F-]+))"
        )
        tokens = []
        pos = 0
        while pos < len(text):
            m = token_spec.match(text, pos)
            if not m:
                pos += 1
                continue
            q, qvar, sym, ident = m.groups()
            if q:
                tokens.append(("QUANT", (q, qvar.lower())))
            elif sym:
                tokens.append((sym, sym))
            elif ident:
                tokens.append(("ID", ident))
            pos = m.end()
        return tokens

    @classmethod
    def parse_formula(cls, text):
        tokens = cls.tokenize(text)
        if not tokens:
            return None
        pos = [0]

        def peek():
            return tokens[pos[0]] if pos[0] < len(tokens) else (None, None)

        def consume():
            tok = peek()
            pos[0] += 1
            return tok

        def parse_iff():
            left = parse_imp()
            while peek()[0] == "↔":
                consume()
                left = FOLNode("IFF", children=[left, parse_imp()])
            return left

        def parse_imp():
            left = parse_xor()
            while peek()[0] == "→":
                consume()
                left = FOLNode("IMP", children=[left, parse_xor()])
            return left

        def parse_xor():
            left = parse_or()
            while peek()[0] == "⊕":
                consume()
                left = FOLNode("XOR", children=[left, parse_or()])
            return left

        def parse_or():
            left = parse_and()
            while peek()[0] == "∨":
                consume()
                left = FOLNode("OR", children=[left, parse_and()])
            return left

        def parse_and():
            left = parse_unary()
            while peek()[0] == "∧":
                consume()
                left = FOLNode("AND", children=[left, parse_unary()])
            return left

        def parse_unary():
            tok_type, tok_val = peek()
            if tok_type == "¬":
                consume()
                return FOLNode("NOT", children=[parse_unary()])
            if tok_type == "QUANT":
                consume()
                q_sym, q_var = tok_val
                return FOLNode("ALL" if q_sym == "∀" else "EXISTS", children=[parse_unary()], var=q_var)
            if tok_type == "(":
                consume()
                node = parse_iff()
                if peek()[0] == ")":
                    consume()
                return node
            if tok_type == "ID":
                consume()
                pred = tok_val.lower()
                args = []
                if peek()[0] == "(":
                    consume()
                    while peek()[0] not in (")", None):
                        if peek()[0] == ",":
                            consume()
                            continue
                        t_type, t_val = consume()
                        if t_type == "ID":
                            args.append(t_val.lower())
                    if peek()[0] == ")":
                        consume()
                return FOLNode("ATOM", val=(pred, tuple(args)))
            consume()
            return FOLNode("ATOM", val=("true_atom", ()))

        return parse_iff()

    @staticmethod
    def collect_signature(nodes):
        preds = set()
        consts = set()
        default_vars = {"x", "y", "z", "w", "u", "v"}

        def visit(n, bvars):
            if n is None:
                return
            if n.kind in ("ALL", "EXISTS"):
                visit(n.children[0], bvars | {n.var})
            elif n.kind == "ATOM":
                p, args = n.val
                preds.add((p, len(args)))
                for a in args:
                    if a not in bvars and a not in default_vars:
                        consts.add(a)
            else:
                for c in n.children:
                    visit(c, bvars)

        for node in nodes:
            visit(node, set())
        return preds, consts

    @classmethod
    def align_node_with_kb(cls, node, kb_preds, kb_consts):
        if node is None:
            return None
        if node.kind == "ATOM":
            p, args = node.val
            new_args = []
            for a in args:
                if a in kb_consts or a in {"x", "y", "z", "w", "u", "v"}:
                    new_args.append(a)
                else:
                    best_c, best_score = a, 0.0
                    for kc in sorted(kb_consts):
                        sc = SequenceMatcher(None, a, kc).ratio()
                        if sc > best_score:
                            best_c, best_score = kc, sc
                    new_args.append(best_c if best_score >= 0.72 else a)

            if (p, len(new_args)) not in kb_preds:
                best_p, best_score = p, 0.0
                for kp, karity in sorted(kb_preds):
                    if karity == len(new_args):
                        sc = SequenceMatcher(None, p, kp).ratio()
                        if sc > best_score:
                            best_p, best_score = kp, sc
                if best_score >= 0.68:
                    p = best_p
            return FOLNode("ATOM", val=(p, tuple(new_args)))
        return FOLNode(node.kind, node.val, [cls.align_node_with_kb(c, kb_preds, kb_consts) for c in node.children], node.var)

    @classmethod
    def ground_formula(cls, node, domain, env=None, q_depth=0):
        """Grounding with dynamic domain cap to prevent nested quantifier blowup."""
        if env is None:
            env = {}
        if node is None:
            return None

        if node.kind == "ATOM":
            p, args = node.val
            return ("ATOM", (p, tuple(env.get(a, a) for a in args)))
        if node.kind == "NOT":
            return ("NOT", cls.ground_formula(node.children[0], domain, env, q_depth))
        if node.kind in ("AND", "OR", "IMP", "IFF", "XOR"):
            return (
                node.kind,
                cls.ground_formula(node.children[0], domain, env, q_depth),
                cls.ground_formula(node.children[1], domain, env, q_depth),
            )
        if node.kind in ("ALL", "EXISTS"):
            # Cap branching factor for nested quantifiers (depth >= 1 -> max 2 constants)
            active_dom = domain[:3] if q_depth == 0 else domain[:2]
            sub_nodes = [
                cls.ground_formula(node.children[0], domain, {**env, node.var: c}, q_depth + 1)
                for c in active_dom
            ]
            op = "AND" if node.kind == "ALL" else "OR"
            res = sub_nodes[0]
            for s in sub_nodes[1:]:
                res = (op, res, s)
            return res
        return None

    @classmethod
    def tseitin_cnf(cls, expr):
        """Memoized Tseitin CNF conversion returning (clauses, atom_var_ids)."""
        clauses = []
        atom_to_id = {}
        memo = {}
        next_id = [1]

        def get_atom_id(atom):
            if atom not in atom_to_id:
                atom_to_id[atom] = next_id[0]
                next_id[0] += 1
            return atom_to_id[atom]

        def new_aux():
            v = next_id[0]
            next_id[0] += 1
            return v

        def encode(e):
            if e in memo:
                return memo[e]
            k = e[0]
            if k == "ATOM":
                res = get_atom_id(e[1])
                memo[e] = res
                return res
            if k == "NOT":
                res = -encode(e[1])
                memo[e] = res
                return res
            a = encode(e[1])
            b = encode(e[2])
            z = new_aux()
            if k == "AND":
                clauses.extend([[-z, a], [-z, b], [z, -a, -b]])
            elif k == "OR":
                clauses.extend([[z, -a], [z, -b], [-z, a, b]])
            elif k == "IMP":
                clauses.extend([[z, a], [z, -b], [-z, -a, b]])
            elif k == "IFF":
                clauses.extend([[-z, -a, b], [-z, a, -b], [z, a, b], [z, -a, -b]])
            elif k == "XOR":
                clauses.extend([[-z, a, b], [-z, -a, -b], [z, -a, b], [z, a, -b]])
            memo[e] = z
            return z

        top_lit = encode(expr)
        clauses.append([top_lit])
        return clauses, set(atom_to_id.values())

    @classmethod
    def dpll_sat(cls, clauses, atom_vars, assignment=None, stats=None):
        """Bounded DPLL SAT solver that branches strictly on ground atom variables."""
        if assignment is None:
            assignment = {}
        if stats is None:
            stats = [0]

        stats[0] += 1
        if stats[0] > 100:
            # Step budget reached: abort search and treat as satisfiable (non-entailed)
            return True

        # Unit Propagation
        changed = True
        while changed:
            changed = False
            new_clauses = []
            for c in clauses:
                rem = []
                satisfied = False
                for lit in c:
                    var = abs(lit)
                    if var in assignment:
                        if assignment[var] == (lit > 0):
                            satisfied = True
                            break
                    else:
                        rem.append(lit)
                if satisfied:
                    continue
                if not rem:
                    return False  # Empty clause derived (Contradiction!)
                if len(rem) == 1:
                    u_lit = rem[0]
                    u_var = abs(u_lit)
                    u_val = (u_lit > 0)
                    if u_var in assignment and assignment[u_var] != u_val:
                        return False
                    assignment[u_var] = u_val
                    changed = True
                else:
                    new_clauses.append(rem)
            clauses = new_clauses

        if not clauses:
            return True

        # Prefer branching on unassigned Ground Atom variables (Tseitin gates propagate automatically)
        chosen_lit = None
        for c in clauses:
            for lit in c:
                if abs(lit) in atom_vars and abs(lit) not in assignment:
                    chosen_lit = lit
                    break
            if chosen_lit is not None:
                break

        if chosen_lit is None:
            chosen_lit = clauses[0][0]

        var = abs(chosen_lit)
        for val in (chosen_lit > 0, chosen_lit < 0):
            new_assign = dict(assignment)
            new_assign[var] = val
            if cls.dpll_sat(clauses, atom_vars, new_assign, stats):
                return True
        return False

    @classmethod
    def prove_folio(cls, premises_fol_str, conclusion_fol_str):
        if not premises_fol_str or not conclusion_fol_str:
            return False
        prem_lines = [l.strip() for l in premises_fol_str.split("\n") if l.strip()]
        prem_nodes = []
        for l in prem_lines:
            try:
                ast = cls.parse_formula(l)
                if ast:
                    prem_nodes.append(ast)
            except Exception:
                pass

        try:
            conc_node = cls.parse_formula(conclusion_fol_str)
        except Exception:
            return False
        if not prem_nodes or conc_node is None:
            return False

        kb_preds, kb_consts = cls.collect_signature(prem_nodes)
        conc_node = cls.align_node_with_kb(conc_node, kb_preds, kb_consts)
        _, aligned_conc_consts = cls.collect_signature([conc_node])

        # Prioritize constants mentioned in the conclusion first, then KB constants (max 3)
        priority_consts = sorted(aligned_conc_consts) + [c for c in sorted(kb_consts) if c not in aligned_conc_consts]
        domain = priority_consts[:3] if priority_consts else ["c_default"]

        ground_prems = [cls.ground_formula(p, domain) for p in prem_nodes]
        ground_neg_conc = ("NOT", cls.ground_formula(conc_node, domain))

        combined = ground_prems[0]
        for gp in ground_prems[1:]:
            combined = ("AND", combined, gp)

        refutation_formula = ("AND", combined, ground_neg_conc)
        cnf_clauses, atom_vars = cls.tseitin_cnf(refutation_formula)

        # If KB ∧ ¬Conclusion is UNSAT, check that KB itself is satisfiable
        if not cls.dpll_sat(cnf_clauses, atom_vars):
            kb_clauses, kb_atom_vars = cls.tseitin_cnf(combined)
            if cls.dpll_sat(kb_clauses, kb_atom_vars):
                return True
        return False

## 5. Natural-Language Parsing

`NaturalLanguageKBParser` converts supported sentence patterns into facts and rules.

- `parse_ruletaker`: reads `Facts:` and `Rules:` and handles five conditional phrasings.
- `parse_general_nl`: handles simple facts, conditional rules and universal statements.

Text is normalized before parsing; variables are shown as `?x`.

In [17]:
# =====================================================================
# 5. NATURAL LANGUAGE PARSERS (RULETAKER & GENERAL NL KBs)
# =====================================================================

class NaturalLanguageKBParser:
    """
    Converts Natural Language Knowledge Bases + Queries into Internal Representation
    (Facts + HornRules) for RuleTaker and general NL syllogisms.
    """
    RULETAKER_PATTERNS = [
        re.compile(r"^(?:based on|assuming|when|if|given that)\s+(.+?),\s*(?:it is evident that|we can say|we can conclude|then|it follows that)\s+(.+?)\.?$", re.IGNORECASE),
    ]

    @staticmethod
    def normalize_prop(text):
        text = text.strip().rstrip(".!?").strip().lower()
        text = re.sub(r"\s+", " ", text)
        return text

    @classmethod
    def parse_ruletaker(cls, context, statement):
        facts = set()
        rules = []

        mode = "facts"
        for line in context.split("\n"):
            line_s = line.strip()
            if not line_s:
                continue
            if line_s.lower().startswith("facts:"):
                mode = "facts"
                continue
            if line_s.lower().startswith("rules:"):
                mode = "rules"
                continue

            if mode == "facts":
                prop = cls.normalize_prop(line_s)
                if prop:
                    facts.add(Atom(prop))
            else:
                matched = False
                for pat in cls.RULETAKER_PATTERNS:
                    m = pat.match(line_s)
                    if m:
                        prem_str = cls.normalize_prop(m.group(1))
                        conc_str = cls.normalize_prop(m.group(2))
                        prems = [Atom(cls.normalize_prop(p)) for p in re.split(r"\s+and\s+", prem_str)]
                        rules.append(HornRule(prems, Atom(conc_str), raw_text=line_s))
                        matched = True
                        break
                if not matched:
                    prop = cls.normalize_prop(line_s)
                    if prop:
                        facts.add(Atom(prop))

        goal = Atom(cls.normalize_prop(statement))
        return RuleReasoningEngine(facts, rules), goal

    @classmethod
    def parse_general_nl(cls, context, query):
        """
        Parses general natural language KBs such as the Assignment PDF example:
        'Bob is young. Bob is kind. If someone is young, then they are smart.
         If someone is smart and kind, then they are successful.'
        """
        # If formatted with 'Facts:' and 'Rules:', delegate to RuleTaker parser
        if "Facts:" in context and "Rules:" in context:
            return cls.parse_ruletaker(context, query)

        facts = set()
        rules = []
        sentences = [s.strip() for s in re.split(r"[.\n]+", context) if s.strip()]

        for sent in sentences:
            # Pattern 1: "If someone is P1 and P2, then they are Q"
            m_univ = re.match(
                r"^If\s+(?:someone|a person|anyone|something)\s+is\s+(.+?),\s*then\s+(?:they|he|she|it)\s+(?:are|is)\s+(.+)$",
                sent, re.IGNORECASE
            )
            if m_univ:
                conds = [c.strip().lower() for c in re.split(r"\s+and\s+", m_univ.group(1))]
                conc = m_univ.group(2).strip().lower()
                var_x = Term("x", is_var=True)
                prem_atoms = [Atom(c, [var_x]) for c in conds]
                rules.append(HornRule(prem_atoms, Atom(conc, [var_x]), raw_text=sent))
                continue

            # Pattern 2: "All P are Q" / "Every P is Q"
            m_all = re.match(r"^(?:All|Every|Each)\s+([a-zA-Z\s]+?)\s+(?:are|is)\s+([a-zA-Z\s]+)$", sent, re.IGNORECASE)
            if m_all:
                p = m_all.group(1).strip().lower().rstrip("s")
                q = m_all.group(2).strip().lower().rstrip("s")
                var_x = Term("x", is_var=True)
                rules.append(HornRule([Atom(p, [var_x])], Atom(q, [var_x]), raw_text=sent))
                continue

            # Pattern 3: Propositional conditional "If A, then B"
            for pat in cls.RULETAKER_PATTERNS:
                m_rt = pat.match(sent)
                if m_rt:
                    p_str = cls.normalize_prop(m_rt.group(1))
                    c_str = cls.normalize_prop(m_rt.group(2))
                    rules.append(HornRule([Atom(p_str)], Atom(c_str), raw_text=sent))
                    break
            else:
                # Pattern 4: Ground fact "Bob is young"
                m_fact = re.match(r"^([A-Za-z0-9_]+)\s+is\s+(?:a\s+|an\s+)?(.+)$", sent.strip(), re.IGNORECASE)
                if m_fact:
                    entity = Term(m_fact.group(1).lower(), is_var=False)
                    preds = [p.strip().lower().rstrip("s") for p in re.split(r"\s+and\s+", m_fact.group(2))]
                    for p in preds:
                        facts.add(Atom(p, [entity]))
                else:
                    facts.add(Atom(cls.normalize_prop(sent)))

        # Parse Query
        q_clean = query.strip().rstrip(".!?")
        m_q = re.match(r"^([A-Za-z0-9_]+)\s+is\s+(?:a\s+|an\s+)?(.+)$", q_clean, re.IGNORECASE)
        if m_q:
            goal = Atom(m_q.group(2).strip().lower().rstrip("s"), [Term(m_q.group(1).lower(), is_var=False)])
        else:
            goal = Atom(cls.normalize_prop(q_clean))

        return RuleReasoningEngine(facts, rules), goal

## 6. Pipeline and Evaluation

`predict_record(record, fallback_id)` selects the engine from the input fields:

- `given_info`, `question`: CounterBench
- `premises`, `conclusion`: FOLIO (NL fallback if needed)
- `context`, `statement`: RuleTaker

Returns `(id, predicted_bool, gold_bool_or_None)`. The final cell runs the example, evaluates the datasets and writes `submission.csv`.


In [18]:
# =====================================================================
# 6. UNIFIED REASONING PIPELINE & BENCHMARK EVALUATION
# =====================================================================

def predict_record(record, fallback_id=0):
    """
    Routes any dataset record (RuleTaker, CounterBench, FOLIO, or General NL)
    through the appropriate Classical Reasoning Engine and returns:
    (record_id, predicted_bool, ground_truth_bool_or_None)
    """
    rec_id = record.get("id", fallback_id)  # unique per file; question_id/example_id repeat across files

    # Case 1: CounterBench schema ('given_info' and 'question')
    if "given_info" in record and "question" in record:
        engine = CausalGraphEngine()
        pred = engine.solve(record["given_info"], record["question"])
        gt = None
        if "answer" in record:
            gt = str(record["answer"]).strip().lower() in ("yes", "true", "1")
        return rec_id, pred, gt

    # Case 2: FOLIO schema ('premises' and 'conclusion')
    if "premises" in record and "conclusion" in record:
        if "premises-FOL" in record and "conclusion-FOL" in record:
            pred = FOLIOProver.prove_folio(record["premises-FOL"], record["conclusion-FOL"])
        else:
            engine, goal = NaturalLanguageKBParser.parse_general_nl(record["premises"], record["conclusion"])
            pred = engine.forward_chaining(goal) or engine.backward_chaining(goal)
        gt = None
        if "label" in record:
            # In logical entailment, 'True' is entailed (True); 'False' and 'Uncertain' are not entailed (False)
            gt = str(record["label"]).strip().lower() == "true"
        return rec_id, pred, gt

    # Case 3: RuleTaker / General schema ('context' and 'statement' or 'query')
    context = record.get("context", "")
    statement = record.get("statement", record.get("query", ""))
    engine, goal = NaturalLanguageKBParser.parse_general_nl(context, statement)
    pred = engine.forward_chaining(goal)
    gt = None
    if "flag" in record:
        gt = str(record["flag"]).strip().lower() in ("true", "yes", "1")
    elif "label" in record:
        gt = str(record["label"]).strip().lower() in ("true", "yes", "1")
    return rec_id, pred, gt


# --- 6.1 Sanity Check on Problem Statement Example (Bob) ---
bob_context = """
Bob is young.
Bob is kind.
If someone is young, then they are smart.
If someone is smart and kind, then they are successful.
"""
bob_query = "Bob is successful."
bob_engine, bob_goal = NaturalLanguageKBParser.parse_general_nl(bob_context, bob_query)
bob_result = bob_engine.forward_chaining(bob_goal)
print("=== Problem Statement Sanity Check ===")
print(f"Query: {bob_query} -> Prediction: {bob_result}")
for step in bob_engine.trace:
    print("  ", step)
print()


# --- 6.2 Evaluate All Public Datasets & Generate submission.csv ---
# Folder holding the .jsonl files (submission/Data); falls back to the notebook folder
DATA_DIR = "Data" if os.path.isdir("Data") else "."
dataset_files = [
    "ruletaker_train.jsonl",
    "ruletaker_test.jsonl",
    "ruletaker_ood_test.jsonl",
    "counterbench_alpha.jsonl",
    "FOLIO_train.jsonl",
    "FOLIO_validation.jsonl",
]

submission_rows = []
overall_correct = 0
overall_total = 0
overall_start = time.perf_counter()

print(f"{'Dataset':<28} | {'Queries':>8} | {'Correct':>8} | {'Accuracy':>9} | {'Time (s)':>9}")
print("-" * 74)

for filepath in dataset_files:
    path = os.path.join(DATA_DIR, filepath)
    if not os.path.exists(path):
        print(f"WARNING: {path} not found, skipping")
        continue
    ds_correct = 0
    ds_total = 0
    t0 = time.perf_counter()

    with open(path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            rec_id, pred, gt = predict_record(record, fallback_id=f"{filepath}_{idx}")
            ds_total += 1
            if gt is not None and pred == gt:
                ds_correct += 1
            submission_rows.append({
                "id": rec_id,
                "predicted_label": "True" if pred else "False"
            })

    dt = time.perf_counter() - t0
    acc = (ds_correct / ds_total * 100.0) if ds_total > 0 else 0.0
    overall_correct += ds_correct
    overall_total += ds_total
    print(f"{filepath:<28} | {ds_total:>8} | {ds_correct:>8} | {acc:>8.2f}% | {dt:>8.4f}s")

overall_time = time.perf_counter() - overall_start
overall_acc = (overall_correct / overall_total * 100.0) if overall_total > 0 else 0.0
print("-" * 74)
print(f"{'OVERALL TOTAL':<28} | {overall_total:>8} | {overall_correct:>8} | {overall_acc:>8.2f}% | {overall_time:>8.4f}s")

# Save submission.csv
sub_df = pd.DataFrame(submission_rows)
sub_df.to_csv("submission.csv", index=False)
print(f"\nSaved submission.csv with {len(sub_df)} predictions.")

=== Problem Statement Sanity Check ===
Query: Bob is successful. -> Prediction: True
   Initial Facts: {kind(bob), young(bob)}
   Derived smart(bob) from young(bob) via (young(?x) → smart(?x))
   Derived successful(bob) from smart(bob) ∧ kind(bob) via (smart(?x) ∧ kind(?x) → successful(?x))

Dataset                      |  Queries |  Correct |  Accuracy |  Time (s)
--------------------------------------------------------------------------
ruletaker_train.jsonl        |     1000 |      983 |    98.30% |   0.1157s
ruletaker_test.jsonl         |      250 |      244 |    97.60% |   0.0288s
ruletaker_ood_test.jsonl     |      375 |      373 |    99.47% |   0.0446s
counterbench_alpha.jsonl     |     1000 |      960 |    96.00% |   0.0746s
FOLIO_train.jsonl            |     1001 |      774 |    77.32% |   1.0559s
FOLIO_validation.jsonl       |      203 |      165 |    81.28% |   0.2270s
--------------------------------------------------------------------------
OVERALL TOTAL                |  

## Results on the Public Datasets

| Dataset split | Queries | Correct | Accuracy |
|---|---:|---:|---:|
| `ruletaker_train` | 1,000 | 983 | 98.30% |
| `ruletaker_test` | 250 | 244 | 97.60% |
| `ruletaker_ood_test` | 375 | 373 | 99.47% |
| `counterbench_alpha` | 1,000 | 960 | 96.00% |
| `FOLIO_train` | 1,001 | 774 | 77.32% |
| `FOLIO_validation` | 203 | 165 | 81.28% |
| **Total** | **3,829** | **3,499** | **91.38%** |

Total execution time is about 1.3 s on one machine (see the printed table for your run; times vary slightly). Run-to-run results are identical.

**Observations**

* **RuleTaker:** all 25 errors are false positives: the query is derivable from the rules but the dataset label is `False` (most likely label noise in the data; not verified).
* **CounterBench by type:** basic 250/250, conditional 230/250, joint 240/250, nested 240/250.
* **FOLIO:** always answering `False` scores 61.2% (train) and 64.5% (validation). The prover recovers 49% / 57% of the true entailments and 95% of the non-entailed cases, because the domain cap and the 100-node DPLL budget bias undecided cases towards `False`. Strict three-way matching (which never predicts `Uncertain`) would give about 46% / 50%.

**Limitations:** closed-world reasoning cannot represent `Uncertain`; the FOLIO prover is incomplete by design (bounded domain and search); unobserved parentless causal variables are assumed `False`; observations of derived causal variables are not used.